In [2]:
# =========================================================
# FINAL PUBLICATION-QUALITY HYBRID FRAUD DETECTION SYSTEM
# Apriori + Autoencoder + FAI + XGBoost
# =========================================================

# =========================================================
# CLEAN SETUP
# =========================================================
import os
import warnings
import logging

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')

logging.getLogger('tensorflow').setLevel(logging.ERROR)

# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install mlxtend xgboost --quiet

# =========================================================
# IMPORT LIBRARIES
# =========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import (
    LabelEncoder,
    MinMaxScaler
)

from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    train_test_split
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve
)

from xgboost import XGBClassifier

from mlxtend.frequent_patterns import (
    apriori,
    association_rules
)

import tensorflow as tf

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

# =========================================================
# LOAD DATA
# =========================================================
print("\nLoading datasets...")

inpatient = pd.read_csv("/content/Insurance_Fraud_Dataset/Train_Inpatientdata-1542865627584.csv")
outpatient = pd.read_csv("/content/Insurance_Fraud_Dataset/Train_Outpatientdata-1542865627584.csv")
beneficiary = pd.read_csv("/content/Insurance_Fraud_Dataset/Train_Beneficiarydata-1542865627584.csv")
labels_df = pd.read_csv("/content/Insurance_Fraud_Dataset/Train-1542865627584.csv")
# =========================================================
# MERGE DATASETS
# =========================================================
claims = pd.concat([
    inpatient,
    outpatient
])

df = pd.merge(
    claims,
    beneficiary,
    on="BeneID",
    how="left"
)

df = pd.merge(
    df,
    labels_df,
    on="Provider",
    how="left"
)

print("\nDataset Shape:", df.shape)

# =========================================================
# PREPROCESSING + FEATURE ENGINEERING
# =========================================================
print("\nEngineering features...")

work = df.copy()

# =========================================================
# DATE FEATURES
# =========================================================
work['AdmissionDt'] = pd.to_datetime(
    work['AdmissionDt'],
    errors='coerce'
)

work['DischargeDt'] = pd.to_datetime(
    work['DischargeDt'],
    errors='coerce'
)

work['claim_duration'] = (
    work['DischargeDt']
    -
    work['AdmissionDt']
).dt.days

# =========================================================
# CHRONIC CONDITION COUNT
# =========================================================
chronic_cols = [
    col for col in work.columns
    if 'ChronicCond' in col
]

work['chronic_count'] = work[
    chronic_cols
].apply(
    lambda x: (x == 1).sum(),
    axis=1
)

# =========================================================
# PROVIDER CLAIM COUNT
# =========================================================
provider_claims = work.groupby(
    'Provider'
).size()

work['provider_claim_count'] = work[
    'Provider'
].map(provider_claims)

# =========================================================
# BENEFICIARY CLAIM COUNT
# =========================================================
bene_claims = work.groupby(
    'BeneID'
).size()

work['bene_claim_count'] = work[
    'BeneID'
].map(bene_claims)

# =========================================================
# SELECT FEATURES
# =========================================================
features = [
    'Provider',
    'AttendingPhysician',
    'DiagnosisGroupCode',
    'ClmProcedureCode_1',
    'InscClaimAmtReimbursed',
    'DeductibleAmtPaid',
    'IPAnnualReimbursementAmt',
    'OPAnnualReimbursementAmt',
    'IPAnnualDeductibleAmt',
    'OPAnnualDeductibleAmt',
    'claim_duration',
    'chronic_count',
    'provider_claim_count',
    'bene_claim_count'
]

work = work[features]

# =========================================================
# LABELS
# =========================================================
labels = df['PotentialFraud'].map({
    'Yes':1,
    'No':0
})

# =========================================================
# HANDLE MISSING VALUES
# =========================================================
cat_cols = [
    'Provider',
    'AttendingPhysician',
    'DiagnosisGroupCode',
    'ClmProcedureCode_1'
]

num_cols = [
    col for col in work.columns
    if col not in cat_cols
]

cat_imputer = SimpleImputer(
    strategy='most_frequent'
)

num_imputer = SimpleImputer(
    strategy='median'
)

work[cat_cols] = cat_imputer.fit_transform(
    work[cat_cols]
)

work[num_cols] = num_imputer.fit_transform(
    work[num_cols]
)

# =========================================================
# ENCODING
# =========================================================
for col in cat_cols:

    le = LabelEncoder()

    work[col] = le.fit_transform(
        work[col].astype(str)
    )

# =========================================================
# SCALING
# =========================================================
scaler = MinMaxScaler()

work = pd.DataFrame(
    scaler.fit_transform(work),
    columns=work.columns
)

# =========================================================
# SAMPLING
# =========================================================
SAMPLE_SIZE = 100000

sample_idx = np.random.choice(
    len(work),
    SAMPLE_SIZE,
    replace=False
)

work = work.iloc[sample_idx]
labels = labels.iloc[sample_idx]

print("\nSampled Dataset Shape:", work.shape)

# =========================================================
# FEATURES + LABELS
# =========================================================
X = work.values
y = labels.values

# =========================================================
# TRAIN TEST SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =========================================================
# TRAIN ONLY ON NORMAL CLAIMS
# =========================================================
X_train_normal = X_train[
    y_train == 0
]

# =========================================================
# APRIORI RULE MINING
# =========================================================
print("\nRunning Apriori...")

train_df = pd.DataFrame(
    X_train,
    columns=work.columns
)

train_df['diagnosis_bin'] = (
    train_df['DiagnosisGroupCode'] * 100
).astype(int).astype(str)

train_df['procedure_bin'] = (
    train_df['ClmProcedureCode_1'] * 100
).astype(int).astype(str)

basket_diag = pd.get_dummies(
    train_df['diagnosis_bin']
)

basket_proc = pd.get_dummies(
    train_df['procedure_bin']
)

basket = pd.concat(
    [basket_diag, basket_proc],
    axis=1
)

frequent_items = apriori(
    basket,
    min_support=0.001,
    use_colnames=True
)

if frequent_items.empty:

    rules = pd.DataFrame()

else:

    rules = association_rules(
        frequent_items,
        metric="confidence",
        min_threshold=0.2
    )

    rules = rules[
        rules['lift'] >= 1.0
    ]

if not rules.empty:

    rules['rds'] = (
        rules['confidence']
        *
        rules['lift']
    )

print("\nNumber of Rules:", len(rules))

# =========================================================
# AUTOENCODER
# =========================================================
print("\nTraining Autoencoder...")

input_dim = X_train.shape[1]

inputs = Input(shape=(input_dim,))

e = Dense(64, activation='relu')(inputs)
e = Dropout(0.2)(e)

e = Dense(32, activation='relu')(e)

encoded = Dense(16, activation='relu')(e)

d = Dense(32, activation='relu')(encoded)
d = Dropout(0.2)(d)

d = Dense(64, activation='relu')(d)

outputs = Dense(
    input_dim,
    activation='sigmoid'
)(d)

autoencoder = Model(
    inputs,
    outputs
)

autoencoder.compile(
    optimizer=Adam(0.001),
    loss='mse'
)

autoencoder.fit(
    X_train_normal,
    X_train_normal,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# =========================================================
# RECONSTRUCTION ERROR
# =========================================================
print("\nCalculating reconstruction error...")

X_train_pred = autoencoder.predict(X_train)
X_test_pred = autoencoder.predict(X_test)

train_re = np.mean(
    np.square(X_train - X_train_pred),
    axis=1
)

test_re = np.mean(
    np.square(X_test - X_test_pred),
    axis=1
)

# =========================================================
# RULE SCORE FUNCTION
# =========================================================
def get_rule_score(dx, pr):

    if rules.empty:
        return 0

    match = rules[
        rules['antecedents']
        .astype(str)
        .str.contains(str(int(dx*100)))
        |
        rules['antecedents']
        .astype(str)
        .str.contains(str(int(pr*100)))
    ]

    if not match.empty:
        return match['rds'].mean()

    return 0

# =========================================================
# RULE SCORES
# =========================================================
print("\nCalculating rule scores...")

train_df_scores = pd.DataFrame(
    X_train,
    columns=work.columns
)

test_df_scores = pd.DataFrame(
    X_test,
    columns=work.columns
)

train_rule_scores = train_df_scores.apply(
    lambda row: get_rule_score(
        row['DiagnosisGroupCode'],
        row['ClmProcedureCode_1']
    ),
    axis=1
)

test_rule_scores = test_df_scores.apply(
    lambda row: get_rule_score(
        row['DiagnosisGroupCode'],
        row['ClmProcedureCode_1']
    ),
    axis=1
)

# =========================================================
# NORMALIZATION
# =========================================================
train_re_norm = (
    train_re - train_re.min()
) / (
    train_re.max() - train_re.min() + 1e-9
)

test_re_norm = (
    test_re - test_re.min()
) / (
    test_re.max() - test_re.min() + 1e-9
)

train_rule_norm = (
    train_rule_scores - train_rule_scores.min()
) / (
    train_rule_scores.max()
    -
    train_rule_scores.min()
    +
    1e-9
)

test_rule_norm = (
    test_rule_scores - test_rule_scores.min()
) / (
    test_rule_scores.max()
    -
    test_rule_scores.min()
    +
    1e-9
)

# =========================================================
# FRAUD ANOMALY INDEX
# =========================================================
alpha = 0.7

train_fai = (
    alpha * train_re_norm
    +
    (1 - alpha) * train_rule_norm
)

test_fai = (
    alpha * test_re_norm
    +
    (1 - alpha) * test_rule_norm
)

print("\nFAI computed successfully.")

# =========================================================
# APPEND FAI FEATURE
# =========================================================
X_train_final = np.column_stack([
    X_train,
    train_fai
])

X_test_final = np.column_stack([
    X_test,
    test_fai
])

# =========================================================
# XGBOOST CLASSIFIER
# =========================================================
print("\nTraining Hybrid XGBoost...")

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

model.fit(
    X_train_final,
    y_train
)

predictions = model.predict(
    X_test_final
)

probabilities = model.predict_proba(
    X_test_final
)[:,1]

# =========================================================
# RESULTS
# =========================================================
print("\n==============================")
print("FINAL HYBRID MODEL RESULTS")
print("==============================")

accuracy = accuracy_score(
    y_test,
    predictions
)

precision = precision_score(
    y_test,
    predictions
)

recall = recall_score(
    y_test,
    predictions
)

f1 = f1_score(
    y_test,
    predictions
)

roc_auc = roc_auc_score(
    y_test,
    probabilities
)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC-AUC  :", roc_auc)

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        predictions
    )
)

# =========================================================
# GRAPH 1 — CONFUSION MATRIX
# =========================================================
cm = confusion_matrix(
    y_test,
    predictions
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title(
    "Confusion Matrix"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

# =========================================================
# GRAPH 2 — ROC CURVE
# =========================================================
fpr, tpr, _ = roc_curve(
    y_test,
    probabilities
)

roc_auc = auc(
    fpr,
    tpr
)

plt.figure(figsize=(7,6))

plt.plot(
    fpr,
    tpr,
    linewidth=3,
    label=f'AUC = {roc_auc:.3f}'
)

plt.plot(
    [0,1],
    [0,1],
    linestyle='--'
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "ROC Curve"
)

plt.legend()

plt.grid(True)

plt.show()

# =========================================================
# GRAPH 3 — PRECISION RECALL CURVE
# =========================================================
precision_curve, recall_curve, _ = precision_recall_curve(
    y_test,
    probabilities
)

plt.figure(figsize=(7,6))

plt.plot(
    recall_curve,
    precision_curve,
    linewidth=3
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title(
    "Precision-Recall Curve"
)

plt.grid(True)

plt.show()

# =========================================================
# GRAPH 4 — FEATURE IMPORTANCE
# =========================================================
feature_names = list(work.columns) + ['FAI']

importance = model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=importance_df.head(10)
)

plt.title(
    "Top Feature Importances"
)

plt.show()

# =========================================================
# GRAPH 5 — FRAUD PROBABILITY DISTRIBUTION
# =========================================================
plt.figure(figsize=(9,5))

sns.histplot(
    probabilities,
    bins=50,
    kde=True
)

plt.title(
    "Fraud Probability Distribution"
)

plt.xlabel("Fraud Probability")
plt.ylabel("Frequency")

plt.show()

# =========================================================
# GRAPH 6 — PERFORMANCE METRICS
# =========================================================
metrics = [
    accuracy,
    precision,
    recall,
    f1,
    roc_auc
]

metric_names = [
    'Accuracy',
    'Precision',
    'Recall',
    'F1',
    'ROC-AUC'
]

plt.figure(figsize=(8,5))

bars = plt.bar(
    metric_names,
    metrics
)

plt.ylim(0,1)

plt.title(
    "Hybrid Model Performance"
)

for bar, value in zip(bars, metrics):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.01,
        f'{value:.2f}',
        ha='center'
    )

plt.show()

# =========================================================
# GRAPH 7 — FAI DISTRIBUTION
# =========================================================
plt.figure(figsize=(9,5))

sns.histplot(
    test_fai,
    bins=50,
    kde=True
)

plt.title(
    "Fraud Anomaly Index Distribution"
)

plt.xlabel("FAI Score")
plt.ylabel("Frequency")

plt.show()

print("\nAll graphs generated successfully.")

# =======================================================
print("\nFirst 10 Test FAI Scores:\n")

print(test_fai[:10])

fai_results = pd.DataFrame({
    'FAI_Score': test_fai,
    'Predicted_Label': predictions,
    'Actual_Label': y_test
})

print(fai_results.head(20))

top_fraud = fai_results.sort_values(
    by='FAI_Score',
    ascending=False
)

print("\nTop Suspicious Claims:\n")

print(top_fraud.head(20))

fai_results.to_csv(
    "FAI_Scores.csv",
    index=False
)

print("\nFAI scores saved successfully.")

print("\nAverage FAI Scores:\n")

print(
    fai_results.groupby(
        'Actual_Label'
    )['FAI_Score'].mean()
)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


Loading datasets...


FileNotFoundError: [Errno 2] No such file or directory: '/content/Insurance_Fraud_Dataset/Train_Inpatientdata-1542865627584.csv'